# Projeto de POO com Python: Calculadora Desktop
**Notebook do aluno — teoria, roteiro de desenvolvimento e distribuição para Windows**

Neste projeto você desenvolverá uma calculadora desktop usando Python, Tkinter e Programação Orientada a Objetos. Ao final, deverá gerar um executável `.exe` para Windows.

## Objetivos
- Identificar classes, objetos, atributos e métodos.
- Aplicar encapsulamento, abstração, herança, polimorfismo e composição.
- Separar interface gráfica da lógica de negócio.
- Tratar entradas inválidas e testar o sistema.
- Empacotar a aplicação com PyInstaller.

> O notebook é o material de estudo. A aplicação final deverá ser organizada em arquivos `.py`.

## 1. Problema e requisitos

Uma organização precisa de uma calculadora desktop simples, com operações matemáticas básicas e histórico de cálculos.

### Requisitos funcionais
| Código | Requisito |
|---|---|
| RF01 | Informar dois números. |
| RF02 | Somar, subtrair, multiplicar e dividir. |
| RF03 | Exibir o resultado. |
| RF04 | Limpar os campos e o resultado. |
| RF05 | Registrar operações no histórico. |
| RF06 | Consultar e limpar o histórico. |
| RF07 | Informar entradas inválidas. |
| RF08 | Impedir divisão por zero. |

### Requisitos não funcionais
- Interface clara e organizada.
- Código organizado em classes com responsabilidades distintas.
- A interface não deve concentrar a lógica matemática.
- O programa deverá ser empacotado para Windows.

**Critério de aceitação:** o usuário consegue realizar as quatro operações, visualizar o resultado, limpar os campos, consultar o histórico e receber mensagens compreensíveis em caso de erro.

## 2. Fundamentos de POO

A Programação Orientada a Objetos organiza o software em objetos. Cada objeto reúne **dados (atributos)** e **comportamentos (métodos)**.

Uma **classe** é um modelo para criar objetos. Um **objeto** é uma instância de uma classe.

### Classe, objeto, atributo e método

```python
class Operacao:
    def __init__(self, primeiro, segundo):
        self.primeiro = primeiro
        self.segundo = segundo

    def executar(self):
        return self.primeiro + self.segundo

operacao = Operacao(10, 5)
print(operacao.executar())  # 15
```

- `Operacao`: classe.
- `primeiro` e `segundo`: atributos de instância.
- `executar()`: método.
- `self`: referência à instância atual.
- `__init__`: método inicializador, chamado ao criar o objeto.

**Reflita:** por que guardar os valores no objeto pode ser melhor do que realizar a conta em uma linha isolada?
> **Resposta:** Guardar valores em um objeto melhora a clareza, a reutilização e a manutenção do código em comparação a fazer o cálculo direto em uma linha isolada.

### 2.1 Encapsulamento

Encapsulamento organiza e controla o acesso ao estado interno do objeto. Em Python, um atributo iniciado por `_`, como `_resultado`, indica por convenção que ele deve ser tratado como interno. Não é uma barreira rígida de segurança.

`@property` permite oferecer uma forma controlada de leitura:

```python
class Resultado:
    def __init__(self, valor):
        self._valor = valor

    @property
    def valor(self):
        return self._valor

item = Resultado(25)
print(item.valor)
```

No projeto, a interface não deverá manipular diretamente a lista interna do histórico: deverá usar métodos da classe `Historico`.

### 2.2 Abstração

Abstrair é destacar o que é essencial e esconder detalhes de implementação. O usuário solicita uma operação sem precisar conhecer seu algoritmo.

Uma classe abstrata define um contrato para subclasses:

```python
from abc import ABC, abstractmethod

class Operacao(ABC):
    @abstractmethod
    def executar(self):
        pass
```

A classe abstrata não deve ser instanciada diretamente.

### 2.3 Herança

Herança permite criar uma classe especializada a partir de outra, reaproveitando atributos e comportamentos.

```python
class Operacao:
    def __init__(self, a, b):
        self.a, self.b = a, b

class Soma(Operacao):
    def executar(self):
        return self.a + self.b
```

### 2.4 Polimorfismo

Polimorfismo permite tratar objetos de classes diferentes por uma interface comum, embora cada classe execute o comportamento de modo próprio.

```python
class Subtracao(Operacao):
    def executar(self):
        return self.a - self.b

operacoes = [Soma(8, 2), Subtracao(8, 2)]
for op in operacoes:
    print(op.executar())
```

O laço chama `executar()` sem precisar conhecer os detalhes de cada classe.

### 2.5 Composição

Composição ocorre quando um objeto utiliza ou contém outro objeto. No projeto, `Calculadora` utilizará um objeto `Historico`. A calculadora não é um histórico; ela utiliza um histórico.

## 3. Modelagem do projeto

| Classe | Responsabilidade |
|---|---|
| `Operacao` | Contrato abstrato das operações. |
| `Soma` | Executar adição. |
| `Subtracao` | Executar subtração. |
| `Multiplicacao` | Executar multiplicação. |
| `Divisao` | Executar divisão e validar divisor. |
| `Historico` | Armazenar e fornecer registros. |
| `Calculadora` | Coordenar execução e registro. |
| `CalculadoraApp` | Criar interface e tratar eventos. |

```text
                   Operacao (abstrata)
              /      |         |         \
           Soma Subtracao Multiplicacao Divisao

       Calculadora ------> Historico
            ^
            |
      CalculadoraApp
      (interface Tkinter)
```

As operações concretas herdam de `Operacao`. `Calculadora` utiliza `Historico`. `CalculadoraApp` utiliza `Calculadora`.

### Separação de responsabilidades
- **Interface:** captura entradas, exibe resultados e mensagens.
- **Regras de negócio:** executa cálculos.
- **Histórico:** mantém os registros.

## 4. Preparação do ambiente e arquivos

Recomendação: Python 3.10 ou superior. Verifique:

```bash
python --version
python -m tkinter
```

Se a janela de demonstração abrir, Tkinter está disponível. A instalação oficial do Python para Windows normalmente inclui Tkinter.

Crie a pasta `CalculadoraPOO` com esta estrutura:

```text
CalculadoraPOO/
├── main.py
├── operacoes.py
├── historico.py
├── calculadora.py
├── interface.py
└── README.md
```

Todos os arquivos `.py` deverão ficar inicialmente na mesma pasta.

## 5. Implementar `operacoes.py`

A classe abstrata define o contrato. As classes concretas implementam `executar()`.

```python
from abc import ABC, abstractmethod

class Operacao(ABC):
    def __init__(self, primeiro, segundo):
        self._primeiro = float(primeiro)
        self._segundo = float(segundo)

    @abstractmethod
    def executar(self):
        pass
```

### Sua tarefa
Implemente `Soma`, `Subtracao`, `Multiplicacao` e `Divisao`, herdando de `Operacao`. Todas devem implementar `executar()`. A divisão deverá lançar `ValueError` se o divisor for zero.

Exemplo de tratamento:

```python
try:
    # executar uma divisão
    pass
except ValueError as erro:
    print("Erro:", erro)
```

Perguntas:
1. Qual a vantagem de todas as operações terem `executar()`?
> **Resposta:** A principal vantagem de todas as operações terem o método `executar()` é permitir o polimorfismo, fazendo com que diferentes ações sejam tratadas de forma uniforme por meio de uma interface ou classe base comum.

2. Que conceito está presente entre `Soma` e `Operacao`?
> **Resposta:** Herança (relação "é um"): `Soma` herda de `Operacao` e especializa o comportamento.

3. Por que a validação de divisor zero pertence à classe `Divisao`?
> **Resposta:** Porque essa regra de negócio é exclusiva da operação de divisão. Manter a validação dentro da própria classe garante coesão e impede que o erro ocorra independentemente de qual interface chame a operação.

## 6. Implementar `historico.py`

Crie a classe `Historico` com uma lista interna de registros e os métodos:
- `adicionar(registro)`: inclui um registro;
- `listar()`: retorna os registros;
- `limpar()`: remove todos os registros.

Estrutura inicial:

```python
class Historico:
    def __init__(self):
        self._registros = []

    def adicionar(self, registro):
        pass

    def listar(self):
        pass

    def limpar(self):
        pass
```

A interface não deve acessar diretamente `_registros`.

## 7. Implementar `calculadora.py`

A classe `Calculadora` deverá criar ou receber um objeto `Historico` e possuir o método `calcular(operacao, simbolo)`.

Esse método deverá:
1. receber um objeto de operação;
2. executar a operação;
3. registrar expressão e resultado;
4. devolver o resultado.

Estrutura inicial:

```python
class Calculadora:
    def __init__(self):
        pass

    def calcular(self, operacao, simbolo):
        pass
```

O método deverá receber um objeto de operação, não uma cadeia extensa de `if/elif` contendo todas as contas. Assim você praticará polimorfismo.

## 8. Tkinter: interface gráfica

Tkinter é a biblioteca padrão do Python para interfaces gráficas simples.

### Janela principal

```python
import tkinter as tk

janela = tk.Tk()
janela.title("Minha Calculadora")
janela.geometry("360x420")
janela.mainloop()
```

- `Tk()` cria a janela.
- `title()` define o título.
- `geometry()` define dimensões iniciais.
- `mainloop()` mantém a janela ativa e processa eventos.

### Componentes

| Componente | Finalidade |
|---|---|
| `Label` | Exibir texto. |
| `Entry` | Receber dados. |
| `Button` | Executar ação. |
| `Frame` | Agrupar componentes. |
| `Listbox` | Exibir lista. |
| `Scrollbar` | Rolagem. |

### Gerenciadores de layout
- `pack()`: organiza componentes em blocos.
- `grid()`: organiza em linhas e colunas.
- `place()`: posiciona por coordenadas.

Evite misturar `pack()` e `grid()` no mesmo contêiner.

### Eventos, callbacks e validação

A interface é orientada a eventos: um clique chama uma função (callback).

```python
def executar_soma():
    print("Solicitação de soma")

botao = tk.Button(janela, text="Somar", command=executar_soma)
```

Use `command=executar_soma`, sem parênteses: a função será chamada quando o botão for clicado.

`Entry.get()` retorna texto. Converta para `float` e trate erros:

```python
texto = entrada.get()
try:
    numero = float(texto)
except ValueError:
    print("Digite um número válido.")
```

Para mostrar erros na interface:

```python
from tkinter import messagebox
messagebox.showerror("Erro", "Digite valores numéricos válidos.")
```

### Sua tarefa: `interface.py`

Crie `CalculadoraApp`, que recebe um objeto `Calculadora` e:
- cria janela e campos para dois números;
- cria botões para as quatro operações;
- exibe o resultado;
- exibe o histórico;
- oferece limpeza de campos e de histórico;
- trata entradas inválidas e divisão por zero.

A interface deverá criar o objeto de operação adequado e solicitar o cálculo à `Calculadora`. Não faça as contas diretamente nos callbacks.

## 9. Ponto de entrada: `main.py`

Conecte as classes e inicie a interface:

```python
from calculadora import Calculadora
from interface import CalculadoraApp

def main():
    calculadora = Calculadora()
    app = CalculadoraApp(calculadora)
    app.executar()

if __name__ == "__main__":
    main()
```

Execute no terminal, dentro da pasta:

```bash
python main.py
```

Se houver erro de importação, confira nomes dos arquivos, classes e a pasta atual.

## 10. Testes

Teste a lógica de negócio separadamente da interface. Exemplo:

```python
# Depois de implementar as classes:
# assert Soma(10, 5).executar() == 15
# assert Subtracao(10, 5).executar() == 5
```

Para divisão por zero, verifique se `ValueError` foi lançado.

### Roteiro mínimo

| Caso | Entrada | Resultado esperado |
|---|---|---|
| Adição | 10 e 5 | 15 |
| Subtração | 10 e 5 | 5 |
| Multiplicação | 10 e 5 | 50 |
| Divisão | 10 e 5 | 2 |
| Divisão por zero | 10 e 0 | Mensagem de erro |
| Entrada inválida | abc e 5 | Mensagem de erro |
| Histórico | Duas operações | Dois registros |
| Limpeza | Acionar limpar | Campos/histórico limpos |

**Reflexão:** por que testar a camada de negócio separadamente da interface facilita encontrar defeitos?
> **Resposta:** Testar a camada de negócio separadamente da interface facilita encontrar defeitos porque isola a lógica central do sistema, permitindo identificar exatamente onde está a falha sem o ruído visual ou técnico da tela.

## 11. Gerar o executável `.exe` para Windows

Utilizaremos o **PyInstaller**. Ele empacota o programa Python e suas dependências em uma distribuição executável.

> Gere o executável no Windows para Windows. PyInstaller não é um compilador cruzado: não conte com gerar um `.exe` Windows a partir de Linux ou macOS.

### 11.1 Criar ambiente virtual

Abra o Prompt de Comando ou terminal do VS Code na pasta do projeto:

```bat
python -m venv .venv
.venv\Scripts\activate
python -m pip install --upgrade pip
python -m pip install pyinstaller
```

No PowerShell, a ativação é:

```powershell
.\.venv\Scripts\Activate.ps1
```

### 11.2 Testar antes de empacotar

```bash
python main.py
```

Resolva os erros antes de gerar o executável.

### 11.3 Gerar o `.exe`

Na pasta que contém `main.py`:

```bash
pyinstaller --onefile --windowed --name CalculadoraPOO main.py
```

Opções:
- `--onefile`: gera um executável principal em arquivo único.
- `--windowed`: não abre uma janela de terminal junto com a interface.
- `--name`: define o nome do executável.

O arquivo final ficará em:

```text
dist/CalculadoraPOO.exe
```

### 11.4 Testar e distribuir
1. Abra a pasta `dist`.
2. Execute `CalculadoraPOO.exe`.
3. Teste todas as funcionalidades.
4. Copie para outro computador Windows e teste novamente.

Para ícone personalizado, use um arquivo `.ico`:

```bash
pyinstaller --onefile --windowed --icon=icone.ico --name CalculadoraPOO main.py
```

### 11.5 Problemas comuns
- Se o `.exe` não abrir, gere uma versão sem `--windowed` para ver mensagens no terminal.
- Se faltar módulo, verifique os imports e se PyInstaller foi instalado no mesmo ambiente.
- Antivírus pode gerar alertas heurísticos para executáveis empacotados; distribua apenas arquivos de origem conhecida.
- `--onefile` pode extrair componentes temporários ao iniciar e não garante compatibilidade com toda versão do Windows.

## 12. Entrega e apresentação

Entregue:
1. arquivos `.py` do projeto;
2. notebook preenchido;
3. diagrama de classes;
4. `CalculadoraPOO.exe`;
5. `README.md` com instruções e funcionalidades;
6. demonstração do programa.

### Perguntas para apresentação

#### 1. Quais classes foram criadas e por quê?
Foram criadas 8 classes para separar responsabilidades e aplicar os conceitos de POO do projeto:
- **`Operacao`**: define o contrato abstrato (`executar()`) e armazena os dois operandos (`_primeiro` e `_segundo`). Evita repetição de código e permite tratar operações de forma uniforme.
- **`Soma`**: herda de `Operacao` e executa a adição.
- **`Subtracao`**: herda de `Operacao` e executa a subtração.
- **`Multiplicacao`**: herda de `Operacao` e executa a multiplicação.
- **`Divisao`**: herda de `Operacao`, executa a divisão e valida se o divisor é zero (lançando `ValueError`), mantendo essa regra na própria operação.
- **`Historico`**: encapsula a lista interna de registros e expõe métodos (`adicionar()`, `listar()`, `limpar()`) para controlar o acesso aos dados.
- **`Calculadora`**: coordena a execução da operação recebida (`calcular()`) e registra o resultado no `Historico` (composição), sem depender de longas cadeias de `if/elif`.
- **`CalculadoraApp`**: cria a interface gráfica com Tkinter, captura entradas, trata erros de interface/validação e delega os cálculos para `Calculadora`, mantendo a tela separada da lógica de negócio.

#### 2. Onde aparecem encapsulamento, abstração, herança e polimorfismo?
- **Abstração:** Na classe base `Operacao` (em `operacoes.py`), que utiliza o módulo `abc` (`ABC` e `@abstractmethod`) para definir um contrato genérico (`executar()`), focando no essencial e impedindo instanciação direta.
- **Herança:** Nas classes `Soma`, `Subtracao`, `Multiplicacao` e `Divisao`, que herdam de `Operacao` reaproveitando o construtor (`__init__`) e os atributos `_primeiro` e `_segundo`.
- **Polimorfismo:** Na interação da classe `Calculadora` com os objetos de operação, executando qualquer operação via `.executar()` sem precisar conhecer qual subclasse está sendo processada.
- **Encapsulamento:** Principalmente na classe `Historico`, que protege a lista interna `_registros` e expõe apenas os métodos controlados `adicionar()`, `listar()` e `limpar()`.

#### 3. Como a interface conversa com a lógica de negócio?
A interface (`CalculadoraApp`, construída com Tkinter) conversa com a lógica de negócio através de delegação, chamadas de métodos e eventos (callbacks). Ela lê e valida os dados digitados, instancia o objeto de operação correspondente e envia para `Calculadora.calcular()`, exibindo apenas o resultado ou mensagens de erro.

#### 4. Como o histórico é armazenado?
O histórico é armazenado de forma encapsulada em memória RAM, utilizando uma lista interna (`self._registros`) gerenciada exclusivamente pelos métodos da classe `Historico`.

#### 5. Quais erros foram tratados?
1. **Entradas Inválidas (Erros de Digitação):** Na interface (`CalculadoraApp`), a conversão dos textos para `float` é protegida por um bloco `try...except ValueError`, interrompendo o cálculo e exibindo um alerta (`messagebox.showerror`) caso o usuário digite letras ou deixe campos vazios.
2. **Divisão por Zero:** Na classe de negócio `Divisao`, verifica-se se o divisor é zero antes de calcular, lançando `raise ValueError(...)`, que é capturado pela interface para exibir uma mensagem amigável ao usuário.

#### 6. Como foi gerado o `.exe`?
O executável `.exe` foi gerado utilizando a ferramenta **PyInstaller** via linha de comando no terminal do Windows (`pyinstaller --onefile --windowed --name CalculadoraPOO main.py`), empacotando o código e as dependências em um único arquivo na pasta `dist/`.

### Desafios opcionais
- potenciação e raiz quadrada;
- operações encadeadas;
- exportação do histórico para `.txt` ou `.csv`;
- persistência do histórico;
- calculadora científica;
- testes automatizados com `unittest`.

### Autoavaliação
- [x] Sei diferenciar classe e objeto.
- [x] Identifico atributos e métodos.
- [x] Sei explicar `self`.
- [x] Identifico herança e polimorfismo.
- [x] Sei explicar a composição entre calculadora e histórico.
- [x] A interface não contém toda a lógica matemática.
- [x] Testei entradas inválidas e divisão por zero.
- [x] Gerei e testei o executável no Windows.

**Parabéns! Você desenvolveu uma aplicação desktop com POO e percorreu etapas de um projeto de software.**